# Run your fine-tuned model

Copy the model repository into your Hugging Face account first. Set `MODEL_ID` below to `your-account/your-model`. For a private repository, sign in from the notebook with a Hugging Face access token. A compatible GPU and enough memory for your model are required; 70B models need substantial GPU memory.

In [ ]:
%pip install -q torch transformers accelerate huggingface_hub

In [ ]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "your-account/your-model"
# Uncomment for a private repository. The notebook prompts for your token.
# login()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
)
print(f"Loaded {MODEL_ID} on {model.device}")

In [ ]:
def chat(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
        )
    response = output[0, inputs.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

print(chat("Hello. What can you help me with?"))